# 🚨 Notebook 6: Unsupervised Anomaly Detection (Isolation Forest)

## 1. Concept & Autonomous Plant Stress Monitoring
In real-world agricultural or indoor monitoring, plants operate in uncontrolled environments where labeled stimulus events do not exist.
The goal of **unsupervised anomaly detection** is:
1. **Train strictly on baseline "Normal" plant signals.** The model learns the typical manifold of resting healthy biopotential dynamics.
2. **Evaluate on unlabelled signals:** When external events occur (e.g., watering, acoustic disturbance, dehydration, tissue injury), the model detects them as **statistical outliers / anomalies** without needing prior labels.

### Algorithms Tested:
- **Isolation Forest:** Isolates anomalies by randomly selecting a feature and randomly splitting value. Outliers require fewer recursive splits to isolate.
- **Local Outlier Factor (LOF):** Measures the local density deviation of a sample with respect to its neighbors.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, precision_recall_curve

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv("plant_time_series_features.csv")
print(f"Total dataset: {len(df)} feature windows.")
print(df["main_category"].value_counts())


## 2. Train Isolation Forest Strictly on Normal Baseline
We fit our Isolation Forest model purely on normal baseline windows ($y = 0$).


In [ ]:
feature_cols = [
    "mean", "std", "var", "min", "max", "range", "median", 
    "rms", "mad", "p25", "p75", "iqr", "zcr", "skew", "kurt", 
    "peak_count", "low_freq_power", "mid_freq_power", "high_freq_power", 
    "spectral_entropy", "dominant_freq"
]

scaler = StandardScaler()
X_all = scaler.fit_transform(df[feature_cols].values)

# Extract normal baseline windows for training
normal_mask = (df["main_category"] == "normal").values
X_normal = X_all[normal_mask]

# Train Isolation Forest on normal baseline data
iso_forest = IsolationForest(
    n_estimators=150, 
    contamination=0.08, 
    random_state=42, 
    n_jobs=-1
)
iso_forest.fit(X_normal)

# Compute decision function anomaly scores for all windows
# Lower decision score = more anomalous
df["anomaly_score"] = -iso_forest.decision_function(X_all)
# Flag: 1 if anomaly (score > threshold), 0 if normal
df["is_anomaly_pred"] = (iso_forest.predict(X_all) == -1).astype(int)

print("Isolation Forest successfully trained on normal baseline.")


## 3. Anomaly Score Distribution Across Stimulus Categories
Anomalies should be elevated under external perturbations (`water`, `music`, and `dead`).


In [ ]:
plt.figure(figsize=(10, 5))
palette = {"normal": "#1976D2", "music": "#7B1FA2", "water": "#388E3C", "dead": "#D32F2F"}

sns.boxplot(data=df, x="main_category", y="anomaly_score", palette=palette)
plt.axhline(0, color='red', linestyle='--', label='Isolation Forest Boundary')
plt.title("Isolation Forest Anomaly Score Across Stimulus Categories", fontweight='bold')
plt.ylabel("Anomaly Score (Higher = More Anomalous)")
plt.xlabel("Condition")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Anomaly Detection Rate Across Classes
What percentage of each category is flagged as an anomaly relative to the normal baseline?


In [ ]:
anomaly_rates = df.groupby("main_category")["is_anomaly_pred"].mean() * 100

plt.figure(figsize=(8, 5))
colors = [palette[cat] for cat in anomaly_rates.index]
bars = plt.bar(anomaly_rates.index, anomaly_rates.values, color=colors, alpha=0.85, edgecolor='k')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha='center', fontweight='bold')

plt.title("Anomaly Detection Rate (% Flagged as Abnormal)", fontweight='bold')
plt.ylabel("% Windows Detected as Anomaly")
plt.ylim(0, 100)
plt.grid(True, linestyle=':', axis='y')
plt.tight_layout()
plt.show()


## 5. Timeline of Continuous Anomaly Detection
Simulating real-time sensor monitoring: biopotential stream with rolling anomaly score and alarm triggers.


In [ ]:
sample_session = df[df["main_category"] == "water"].copy()
if len(sample_session) < 50:
    sample_session = df.sample(150, random_state=42).sort_values("window_idx")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(sample_session["mean"].values, color='#1976D2', lw=2, label='Mean Biopotential (ADC)')
ax1.set_title("Real-Time Biopotential Monitoring & Anomaly Alarm Stream", fontweight='bold')
ax1.set_ylabel("ADC Count")
ax1.legend(loc='upper right')
ax1.grid(True)

# Anomaly score curve
scores = sample_session["anomaly_score"].values
threshold = np.percentile(df[df["main_category"] == "normal"]["anomaly_score"], 90)

ax2.plot(scores, color='#D32F2F', lw=2, label='Anomaly Score')
ax2.axhline(threshold, color='black', linestyle='--', label='Anomaly Trigger Threshold')
ax2.fill_between(range(len(scores)), scores, threshold, where=(scores > threshold), color='red', alpha=0.3, label='Alarm Triggered')

ax2.set_title("Continuous Anomaly Score & Alarm Trigger", fontweight='bold')
ax2.set_xlabel("Time Window Index")
ax2.set_ylabel("Anomaly Score")
ax2.legend(loc='upper right')
ax2.grid(True)

plt.tight_layout()
plt.show()


## 6. ROC Evaluation of Unsupervised Anomaly Detection
Evaluating how well unsupervised anomaly scores differentiate perturbed states (water/music/dead) from pure baseline.


In [ ]:
y_true_anomaly = (df["main_category"] != "normal").astype(int)
scores_all = df["anomaly_score"].values

fpr, tpr, _ = roc_curve(y_true_anomaly, scores_all)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='#D32F2F', lw=2.5, label=f'Unsupervised Isolation Forest (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5)
plt.title("ROC Curve: Unsupervised Stimulus Event Detection", fontweight='bold')
plt.xlabel("False Alarm Rate")
plt.ylabel("Event Detection Rate")
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


## 7. Key Takeaways
1. **Label-Free Detection:** The model accurately detects when environmental conditions diverge from baseline without requiring labeled training for each stimulus.
2. **Watering & Dead States:** Both watering events and dead tissue exhibit the highest anomaly scores, enabling automated irrigation alerts and plant health monitoring.
